# Test — Day 3 Bronze (Auto Loader, XML, DLT)

In [0]:
import unittest

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze", "2. Bronze Schema")
CATALOG = dbutils.widgets.get("catalog_name")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

TABLES_WITH_AUDIT = [
    "streets_autoloader", "streets_xml",
    "streets_csv_dlt", "cars_dlt", "telegram_dlt", "node_locations_dlt", "streets_list_dlt",
]


class Day3BronzeTests(unittest.TestCase):

    def test_all_tables_have_rows(self):
        for t in TABLES_WITH_AUDIT:
            with self.subTest(table=t):
                count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{t}").count()
                self.assertGreater(count, 0, f"{t} is empty.")

    def test_all_tables_have_audit_columns_populated(self):
        for t in TABLES_WITH_AUDIT:
            with self.subTest(table=t):
                df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{t}")
                missing = df.filter("load_dt IS NULL OR source IS NULL").count()
                self.assertEqual(missing, 0, f"{t} has {missing} rows missing load_dt/source.")

    def test_street_chunks_combine_to_full_grain_no_dupes(self):
        """
        copyinto (chunk1) + autoloader (chunk3) + xml (chunk4) + dlt (chunk2)
        together should reconstruct streets.csv's full (street_id, date) grain
        with zero overlap — confirms no chunk/table double-counts a source row.
        """
        c1 = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.streets_csv_copyinto").select("street_id", "date")
        c2 = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.streets_csv_dlt").select("street_id", "date")
        c3 = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.streets_autoloader").select("street_id", "date")
        c4 = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.streets_xml").select("street_id", "date")

        combined = c1.unionByName(c2).unionByName(c3).unionByName(c4)
        total = combined.count()
        distinct = combined.distinct().count()
        self.assertEqual(total, distinct,
                          f"Grain not clean across the 4 street bronze tables: "
                          f"{total:,} rows vs {distinct:,} distinct (street_id, date) pairs.")

        source_total = spark.read.option("header", "true") \
            .csv(f"/Volumes/{CATALOG}/raw/landing/streets.csv").count()
        self.assertEqual(total, source_total,
                          f"4 bronze tables sum to {total:,} rows, source has {source_total:,} — "
                          f"a source row is missing from Bronze.")


# COMMAND ----------

if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(Day3BronzeTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 3 bronze tests FAILED — see output above.")
